# 🤖 NLP Task 5: Fine-Tuning DistilBERT for POS Tagging & Chunking

**Assignment:** Fine-tune a Transformer model (BERT/DistilBERT) to perform:
- Part-of-Speech (POS) Tagging
- Chunking (Phrase Detection)

**Dataset:** CoNLL-2000 (via NLTK)
**Model:** DistilBERT (`distilbert-base-uncased`)
**Framework:** Hugging Face Transformers + PyTorch

---
**Pipeline:**
`Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison`

---
## 📦 Cell 1 — Install Libraries

In [1]:
# Install all required libraries
!pip install transformers datasets seqeval torch accelerate -q

  Preparing metadata (setup.py) ... done


---
## 📥 Cell 2 — Import Libraries

In [2]:
import numpy as np
import torch
import nltk
from nltk.corpus import conll2000
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

✅ All libraries imported successfully!
PyTorch version : 2.10.0+cu128
CUDA available  : True


---
## ✅ Task 1: Dataset Selection

**Dataset:** CoNLL-2000 (via NLTK)
**Label Types:** POS Tags + Chunk Tags (IOB2 format)


---
## 📂 Cell 3 — Load & Build Dataset

In [3]:
# ── Download NLTK CoNLL-2000 corpus ──────────────────────────────────────────
nltk.download('conll2000', quiet=True)

# ── Extract tokens, POS tags, Chunk tags from sentences ──────────────────────
def get_sentences(sents):
    tokens_all, pos_all, chunk_all = [], [], []
    for sent in sents:
        tokens_all.append([w for w, p, c in sent])
        pos_all.append([p for w, p, c in sent])
        chunk_all.append([c for w, p, c in sent])
    return {'tokens': tokens_all, 'pos_tags': pos_all, 'chunk_tags': chunk_all}

train_sents = list(conll2000.iob_sents('train.txt'))
test_sents  = list(conll2000.iob_sents('test.txt'))

train_data  = get_sentences(train_sents)
test_data   = get_sentences(test_sents)

# ── Build label lists from ALL splits (train + test) ─────────────────────────
all_pos_tags   = train_data['pos_tags']   + test_data['pos_tags']
all_chunk_tags = train_data['chunk_tags'] + test_data['chunk_tags']

pos_label_list   = sorted(set(t for s in all_pos_tags   for t in s))
chunk_label_list = sorted(set(t for s in all_chunk_tags for t in s))

print(f'✅ POS labels   ({len(pos_label_list)})  : {pos_label_list}')
print(f'✅ Chunk labels ({len(chunk_label_list)}): {chunk_label_list}')

# ── Convert string labels → integer IDs ──────────────────────────────────────
pos_l2i   = {l: i for i, l in enumerate(pos_label_list)}
chunk_l2i = {l: i for i, l in enumerate(chunk_label_list)}

def encode(data):
    return {
        'tokens'    : data['tokens'],
        'pos_tags'  : [[pos_l2i[t]   for t in seq] for seq in data['pos_tags']],
        'chunk_tags': [[chunk_l2i[t] for t in seq] for seq in data['chunk_tags']],
    }

# ── Encode BEFORE creating datasets (avoids .map() KeyError issues) ───────────
train_encoded = encode(train_data)
test_encoded  = encode(test_data)

split_idx = int(len(train_encoded['tokens']) * 0.8)

dataset = DatasetDict({
    'train'     : Dataset.from_dict({k: v[:split_idx] for k, v in train_encoded.items()}),
    'validation': Dataset.from_dict({k: v[split_idx:] for k, v in train_encoded.items()}),
    'test'      : Dataset.from_dict(test_encoded),
})

print('\n✅ Dataset ready!')
print(dataset)
print(f'\nSample tokens     : {dataset["train"][0]["tokens"][:6]}')
print(f'Sample pos_tags   : {dataset["train"][0]["pos_tags"][:6]}')
print(f'Sample chunk_tags : {dataset["train"][0]["chunk_tags"][:6]}')

✅ POS labels   (44)  : ['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']
✅ Chunk labels (23): ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']

✅ Dataset ready!
DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 7148
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 1788
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 2012
    })
})

Sample tokens     : ['Confidence', 'in', 'the', 'pound', 'is', 'widely']
Sample pos_tags   : [18, 13, 10,

---
## 🏷️ Cell 4 — Peek at a Sample with Label Names

In [4]:
sample = dataset['train'][0]

# Convert IDs back to label strings for display
pos_id2label   = {i: l for i, l in enumerate(pos_label_list)}
chunk_id2label = {i: l for i, l in enumerate(chunk_label_list)}

print('📌 Sample Sentence:')
print('Tokens     :', sample['tokens'])
print('POS Tags   :', [pos_id2label[t]   for t in sample['pos_tags']])
print('Chunk Tags :', [chunk_id2label[t] for t in sample['chunk_tags']])

📌 Sample Sentence:
Tokens     : ['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.']
POS Tags   : ['NN', 'IN', 'DT', 'NN', 'VBZ', 'RB', 'VBN', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NN', 'NNS', 'IN', 'NNP', ',', 'JJ', 'IN', 'NN', 'NN', ',', 'VB', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NNP', 'CC', 'NNP', 'POS', 'JJ', 'NNS', '.']
Chunk Tags : ['B-NP', 'B-PP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'I-VP', 'I-VP', 'I-VP', 'B-NP', 'I-NP', 'I-NP', 'B-SBAR', 'B-NP', 'I-NP', 'B-PP', 'B-NP', 'O', 'B-ADJP', 'B-PP', 'B-NP', 'B-NP', 'O', 'B-VP', 'I-VP', 'I-VP', 'B-NP', 'I-NP', 'I-NP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'I-NP', 'O']


---
## ✅ Task 2 & 3: Configuration + Model Choice

---
## ⚙️ Cell 5 — Select Task & Model

In [5]:
# ── Set TASK = 'pos' for POS Tagging  |  'chunk' for Chunking ────────────────
TASK             = 'chunk'                  # ← change to 'pos' if needed
MODEL_CHECKPOINT = 'distilbert-base-uncased'

if TASK == 'pos':
    label_list = pos_label_list
    tag_column = 'pos_tags'
    task_title = 'POS Tagging'
else:
    label_list = chunk_label_list
    tag_column = 'chunk_tags'
    task_title = 'Chunking (Phrase Detection)'

num_labels = len(label_list)

print(f'✅ Task Selected : {task_title}')
print(f'   Model         : {MODEL_CHECKPOINT}')
print(f'   Tag column    : {tag_column}')
print(f'   Num labels    : {num_labels}')

✅ Task Selected : Chunking (Phrase Detection)
   Model         : distilbert-base-uncased
   Tag column    : chunk_tags
   Num labels    : 23


---
## 🔤 Cell 6 — Load Tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f'✅ Tokenizer loaded: {MODEL_CHECKPOINT}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: distilbert-base-uncased


---
## 🔗 Cell 7 — Tokenization & Label Alignment Function

> **Key Concept:** DistilBERT uses WordPiece tokenization — a word like `"playing"` becomes
> `["play", "##ing"]`. Only the **first subword** gets the real label; the rest get `-100`
> so they are ignored by PyTorch's CrossEntropyLoss.

In [7]:
def tokenize_and_align_labels(examples):
    """
    Tokenizes input sentences and aligns labels with subword tokens.

    Strategy:
    - First subword of each word   → assign the original label
    - Subsequent subwords (##...)  → assign -100  (ignored in loss)
    - [CLS] / [SEP] special tokens → assign -100
    """
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,    # Input is already split into words
        max_length=128,
        padding='max_length',
    )

    all_labels = []
    for i, labels in enumerate(examples[tag_column]):
        word_ids         = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_ids        = []

        for word_id in word_ids:
            if word_id is None:
                # Special tokens [CLS], [SEP], [PAD] → ignore
                label_ids.append(-100)
            elif word_id != previous_word_id:
                # First subword of a new word → assign real label
                label_ids.append(labels[word_id])
            else:
                # Continuation subword (e.g., ##ing) → ignore
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

print('✅ Tokenization & label alignment function defined.')

✅ Tokenization & label alignment function defined.


---
## ⚡ Cell 8 — Apply Tokenization to Full Dataset

In [8]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset['train'].column_names,
)

print('✅ Tokenization complete!')
print(tokenized_datasets)

Map:   0%|          | 0/7148 [00:00<?, ? examples/s]

Map:   0%|          | 0/1788 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]

✅ Tokenization complete!
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7148
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1788
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2012
    })
})


---
## 🔍 Cell 9 — Verify Preprocessing Output

In [9]:
sample_out = tokenized_datasets['train'][0]

print('📊 Preprocessing Output Verification (first 20 tokens):')
print(f'  input_ids      : {sample_out["input_ids"][:20]}')
print(f'  attention_mask : {sample_out["attention_mask"][:20]}')
print(f'  labels         : {sample_out["labels"][:20]}')

print('\n  ▶ Decoded tokens vs labels:')
tokens = tokenizer.convert_ids_to_tokens(sample_out['input_ids'][:20])
for tok, lbl in zip(tokens, sample_out['labels'][:20]):
    lbl_str = label_list[lbl] if lbl != -100 else '[IGNORED]'
    print(f'    {tok:<20} → {lbl_str}')

📊 Preprocessing Output Verification (first 20 tokens):
  input_ids      : [101, 7023, 1999, 1996, 9044, 2003, 4235, 3517, 2000, 2202, 2178, 4629, 11529, 2065, 3119, 4481, 2005, 2244, 1010, 2349]
  attention_mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  labels         : [-100, 5, 6, 5, 16, 10, 21, 21, 21, 21, 5, 16, 16, 8, 5, 16, 6, 5, 22, 0]

  ▶ Decoded tokens vs labels:
    [CLS]                → [IGNORED]
    confidence           → B-NP
    in                   → B-PP
    the                  → B-NP
    pound                → I-NP
    is                   → B-VP
    widely               → I-VP
    expected             → I-VP
    to                   → I-VP
    take                 → I-VP
    another              → B-NP
    sharp                → I-NP
    dive                 → I-NP
    if                   → B-SBAR
    trade                → B-NP
    figures              → I-NP
    for                  → B-PP
    september            → B-NP
    ,             

---
## ✅ Task 3: Model Setup

- `AutoModelForTokenClassification` adds a linear classification head on top of DistilBERT
- `id2label` / `label2id` maps indices ↔ tag strings

---
## 🗂️ Cell 10 — Build Label Mappings

In [10]:
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print('🗂️  id2label mapping:')
for k, v in id2label.items():
    print(f'   {k:>3} → {v}')

🗂️  id2label mapping:
     0 → B-ADJP
     1 → B-ADVP
     2 → B-CONJP
     3 → B-INTJ
     4 → B-LST
     5 → B-NP
     6 → B-PP
     7 → B-PRT
     8 → B-SBAR
     9 → B-UCP
    10 → B-VP
    11 → I-ADJP
    12 → I-ADVP
    13 → I-CONJP
    14 → I-INTJ
    15 → I-LST
    16 → I-NP
    17 → I-PP
    18 → I-PRT
    19 → I-SBAR
    20 → I-UCP
    21 → I-VP
    22 → O


---
## 🤖 Cell 11 — Load DistilBERT Model

In [11]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Model loaded: {MODEL_CHECKPOINT}')
print(f'   Total parameters     : {total_params:,}')
print(f'   Trainable parameters : {trainable_params:,}')
print(f'   Number of labels     : {num_labels}')
print('\n🏗️  Model Architecture (top-level):')
for name, module in list(model.named_children()):
    print(f'   └── {name}: {type(module).__name__}')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded: distilbert-base-uncased
   Total parameters     : 66,380,567
   Trainable parameters : 66,380,567
   Number of labels     : 23

🏗️  Model Architecture (top-level):
   └── distilbert: DistilBertModel
   └── dropout: Dropout
   └── classifier: Linear


---
## ✅ Task 4: Training


---
## 🚀 Cell 12 — Training Arguments

In [12]:
training_args = TrainingArguments(
    output_dir                  = f'./results/{TASK}_model',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    learning_rate               = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    num_train_epochs            = 3,
    weight_decay                = 0.01,
    logging_dir                 = './logs',
    logging_steps               = 100,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    report_to                   = 'none',
    seed                        = 42,
)

print('✅ Training Arguments:')
print(f'   Learning Rate : {training_args.learning_rate}')
print(f'   Epochs        : {training_args.num_train_epochs}')
print(f'   Train Batch   : {training_args.per_device_train_batch_size}')
print(f'   Eval Batch    : {training_args.per_device_eval_batch_size}')
print(f'   Weight Decay  : {training_args.weight_decay}')
print(f'   Output Dir    : {training_args.output_dir}')

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Training Arguments:
   Learning Rate : 2e-05
   Epochs        : 3
   Train Batch   : 16
   Eval Batch    : 16
   Weight Decay  : 0.01
   Output Dir    : ./results/chunk_model


---
## 📐 Cell 13 — Data Collator & Metrics Function

In [13]:
# ── Data Collator: handles dynamic padding per batch ─────────────────────────
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
)

# ── seqeval Metrics ───────────────────────────────────────────────────────────
def compute_metrics(eval_preds):
    """Converts model logits → predicted labels → seqeval P/R/F1 metrics."""
    logits, labels = eval_preds
    predictions    = np.argmax(logits, axis=-1)

    # Remove -100 positions and convert integer IDs → label strings
    true_labels = [
        [label_list[l] for l in label_row if l != -100]
        for label_row in labels
    ]
    true_predictions = [
        [label_list[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]

    return {
        'precision' : precision_score(true_labels, true_predictions),
        'recall'    : recall_score(true_labels, true_predictions),
        'f1'        : f1_score(true_labels, true_predictions),
    }

print('✅ Data Collator and compute_metrics ready.')

✅ Data Collator and compute_metrics ready.


---
## 🏋️ Cell 14 — Initialize Trainer & Train

> ⚠️ Run on **Google Colab T4 GPU** for ~20–25 min training time.

In [14]:
trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized_datasets['train'],
    eval_dataset     = tokenized_datasets['validation'],
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
)

print(f'🚀 Starting training: {task_title}')
print(f'   Train samples      : {len(tokenized_datasets["train"])}')
print(f'   Validation samples : {len(tokenized_datasets["validation"])}')
print('─' * 60)

train_result = trainer.train()

print('─' * 60)
print('✅ Training complete!')
print(f'   Training loss  : {train_result.training_loss:.4f}')
print(f'   Total steps    : {train_result.global_step}')
print(f'   Training time  : {train_result.metrics.get("train_runtime", 0):.1f} sec')

🚀 Starting training: Chunking (Phrase Detection)
   Train samples      : 7148
   Validation samples : 1788
────────────────────────────────────────────────────────────


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.141593,0.119727,0.946283,0.954667,0.950456
2,0.102349,0.100645,0.957189,0.962215,0.959695
3,0.079804,0.097948,0.958493,0.962402,0.960444


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


────────────────────────────────────────────────────────────
✅ Training complete!
   Training loss  : 0.1887
   Total steps    : 1341
   Training time  : 315.8 sec


---
## 💾 Cell 15 — Save Fine-Tuned Model

In [15]:
save_path = f'./fine_tuned_{TASK}_model'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f'✅ Model saved to: {save_path}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to: ./fine_tuned_chunk_model


---
## ✅ Task 5: Evaluation

Using **seqeval** — evaluates full sequences, not just individual tokens.
Reports **Precision**, **Recall**, and **F1 Score** per label + overall.

---
## 📊 Cell 16 — Evaluate on Validation Set

In [16]:
print('📊 Evaluating on Validation Set...')
val_results = trainer.evaluate()

print('\n📈 Validation Results:')
print(f'   Precision : {val_results["eval_precision"]:.4f}')
print(f'   Recall    : {val_results["eval_recall"]:.4f}')
print(f'   F1 Score  : {val_results["eval_f1"]:.4f}')
print(f'   Loss      : {val_results["eval_loss"]:.4f}')

📊 Evaluating on Validation Set...



📈 Validation Results:
   Precision : 0.9585
   Recall    : 0.9624
   F1 Score  : 0.9604
   Loss      : 0.0979


---
## 📈 Cell 17 — Evaluate on Test Set (Full Report)

In [17]:
print('📊 Evaluating on Test Set...')

test_output              = trainer.predict(tokenized_datasets['test'])
logits, labels, metrics_ = test_output
predictions              = np.argmax(logits, axis=-1)

true_labels = [
    [label_list[l] for l in label_row if l != -100]
    for label_row in labels
]
true_predictions = [
    [label_list[p] for p, l in zip(pred_row, label_row) if l != -100]
    for pred_row, label_row in zip(predictions, labels)
]

print('\n📈 Test Set — Detailed Classification Report:')
print('─' * 60)
print(classification_report(true_labels, true_predictions))
print('─' * 60)
print(f'Overall Precision : {precision_score(true_labels, true_predictions):.4f}')
print(f'Overall Recall    : {recall_score(true_labels, true_predictions):.4f}')
print(f'Overall F1 Score  : {f1_score(true_labels, true_predictions):.4f}')

📊 Evaluating on Test Set...



📈 Test Set — Detailed Classification Report:
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

        ADJP       0.81      0.74      0.78       438
        ADVP       0.82      0.85      0.84       866
       CONJP       0.50      0.33      0.40         9
        INTJ       0.00      0.00      0.00         2
         LST       0.00      0.00      0.00         5
          NP       0.96      0.96      0.96     12422
          PP       0.98      0.99      0.98      4811
         PRT       0.69      0.91      0.78       106
        SBAR       0.92      0.94      0.93       535
          VP       0.96      0.96      0.96      4658

   micro avg       0.95      0.96      0.96     23852
   macro avg       0.66      0.67      0.66     23852
weighted avg       0.95      0.96      0.96     23852

────────────────────────────────────────────────────────────
Overall Precision : 0.9542
Overall Recall    : 0.9599
Overall F1 Score  :

---
## ✅ Task 6: Inference

Load the fine-tuned model and predict on **custom sentences**.

---
## 🔍 Cell 18 — Inference via Pipeline

In [18]:
ner_pipeline = pipeline(
    task                 = 'token-classification',
    model                = save_path,
    tokenizer            = save_path,
    aggregation_strategy = 'simple',
)

def predict_tags(sentence, pipeline_obj, task_name='Tag'):
    print(f'\n🔍 Sentence: "{sentence}"')
    print('─' * 65)
    print(f'{"Token":<25} {"Predicted " + task_name:<30} {"Score":>6}')
    print('─' * 65)
    results = pipeline_obj(sentence)
    for entity in results:
        print(f'{entity["word"]:<25} {entity["entity_group"]:<30} {entity["score"]:>6.3f}')
    print('─' * 65)

# ── Run on custom sentences ───────────────────────────────────────────────────
test_sentences = [
    'John works at Google in California.',
    'The quick brown fox jumps over the lazy dog.',
    'Sunita is studying machine learning at IIT Delhi.',
    'Apple released a new iPhone in September 2024.',
]

print(f'\n{"="*65}')
print(f'  INFERENCE RESULTS — {task_title.upper()}')
print(f'{"="*65}')

for sentence in test_sentences:
    predict_tags(sentence, ner_pipeline, task_title.split()[0])

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


  INFERENCE RESULTS — CHUNKING (PHRASE DETECTION)

🔍 Sentence: "John works at Google in California."
─────────────────────────────────────────────────────────────────
Token                     Predicted Chunking              Score
─────────────────────────────────────────────────────────────────
john                      NP                              0.996
works                     VP                              0.998
at                        PP                              0.995
google                    NP                              0.997
in                        PP                              0.997
california                NP                              0.997
─────────────────────────────────────────────────────────────────

🔍 Sentence: "The quick brown fox jumps over the lazy dog."
─────────────────────────────────────────────────────────────────
Token                     Predicted Chunking              Score
──────────────────────────────────────────────────────────────

---
## ✋ Cell 19 — Manual Token-Level Inference

In [19]:
loaded_model     = AutoModelForTokenClassification.from_pretrained(save_path)
loaded_tokenizer = AutoTokenizer.from_pretrained(save_path)
loaded_model.eval()

def predict_manual(sentence):
    """Manual inference showing exact token-level predictions."""
    inputs = loaded_tokenizer(
        sentence,
        return_tensors='pt',
        truncation=True,
        max_length=128,
    )
    with torch.no_grad():
        outputs = loaded_model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)[0]
    tokens      = loaded_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    print(f'\n📌 Sentence: "{sentence}"')
    print(f'{"Token":<20} {"Prediction"}')
    print('-' * 40)
    for tok, pred in zip(tokens, predictions):
        if tok in ['[CLS]', '[SEP]', '[PAD]']:
            continue
        print(f'{tok:<20} {id2label[pred.item()]}')

# Test on two sentences
predict_manual('John works at Google in California .')
predict_manual('The students are learning deep learning at the university .')

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


📌 Sentence: "John works at Google in California ."
Token                Prediction
----------------------------------------
john                 B-NP
works                B-VP
at                   B-PP
google               B-NP
in                   B-PP
california           B-NP
.                    O

📌 Sentence: "The students are learning deep learning at the university ."
Token                Prediction
----------------------------------------
the                  B-NP
students             I-NP
are                  B-VP
learning             I-VP
deep                 B-NP
learning             I-NP
at                   B-PP
the                  B-NP
university           I-NP
.                    O


---
## ✅ Task 7: Comparison — POS Tagging vs Chunking

Comparing the two NLP tasks across multiple dimensions.

---
## ⚖️ Cell 20 — Comparison Table

In [20]:
comparison_data = {
    'Aspect': [
        'Purpose', 'Level of Analysis', 'Label Format',
        'Example Output', 'Difficulty',
        'Label Count (CoNLL-2000)', 'Typical F1 Score', 'Primary Use Cases'
    ],
    'POS Tagging': [
        'Identify grammatical role of each word',
        'Token (word) level',
        'Single flat tag per token (NN, VB, JJ…)',
        'John/NNP works/VBZ at/IN Google/NNP',
        'Easier (grammar-level)',
        f'{len(pos_label_list)} tags',
        '~97%',
        'Parsing, lemmatization, grammar checking'
    ],
    'Chunking': [
        'Group tokens into meaningful phrases',
        'Phrase (multi-token) level',
        'IOB2 format (B-NP, I-NP, B-VP, O…)',
        '[John]/NP [works]/VP [at Google]/PP',
        'Moderate (phrase grouping)',
        f'{len(chunk_label_list)} tags (IOB)',
        '~93%',
        'Information extraction, NER, QA systems'
    ],
}

df = pd.DataFrame(comparison_data).set_index('Aspect')
print('📊 POS Tagging vs Chunking — Side-by-Side Comparison')
print('=' * 95)
print(df.to_string())
print('=' * 95)

📊 POS Tagging vs Chunking — Side-by-Side Comparison
                                                       POS Tagging                                 Chunking
Aspect                                                                                                     
Purpose                     Identify grammatical role of each word     Group tokens into meaningful phrases
Level of Analysis                               Token (word) level               Phrase (multi-token) level
Label Format               Single flat tag per token (NN, VB, JJ…)       IOB2 format (B-NP, I-NP, B-VP, O…)
Example Output                 John/NNP works/VBZ at/IN Google/NNP      [John]/NP [works]/VP [at Google]/PP
Difficulty                                  Easier (grammar-level)               Moderate (phrase grouping)
Label Count (CoNLL-2000)                                   44 tags                            23 tags (IOB)
Typical F1 Score                                              ~97%                  

---
## 🔤 Cell 21 — Visual Example (Same Sentence, Both Tasks)

In [21]:
example_sentence     = ['John', 'works', 'at', 'Google', 'in', 'California']
pos_labels_example   = ['NNP',  'VBZ',   'IN', 'NNP',    'IN', 'NNP']
chunk_labels_example = ['B-NP', 'B-VP',  'B-PP', 'B-NP', 'B-PP', 'B-NP']
phrase_names         = {
    'NP': 'Noun Phrase',
    'VP': 'Verb Phrase',
    'PP': 'Prepositional Phrase'
}

print('🔤 Sentence: John works at Google in California\n')

print('📌 POS Tagging — grammar-level (1 tag per word):')
print(f'  {"Token":<15} {"POS Tag"}')
print('  ' + '-' * 25)
for token, pos in zip(example_sentence, pos_labels_example):
    print(f'  {token:<15} {pos}')

print()
print('📌 Chunking — phrase-level (IOB2 format):')
print(f'  {"Token":<15} {"Chunk Tag":<12} {"Phrase Type"}')
print('  ' + '-' * 42)
for token, chunk in zip(example_sentence, chunk_labels_example):
    phrase = chunk.split('-')[1] if '-' in chunk else 'O'
    print(f'  {token:<15} {chunk:<12} {phrase_names.get(phrase, "")}')

print()
print('🔍 Key Insight:')
print('  POS Tagging → WHAT is each word grammatically? (noun, verb, adjective…)')
print('  Chunking    → HOW do words group into phrases? (NP, VP, PP…)')

🔤 Sentence: John works at Google in California

📌 POS Tagging — grammar-level (1 tag per word):
  Token           POS Tag
  -------------------------
  John            NNP
  works           VBZ
  at              IN
  Google          NNP
  in              IN
  California      NNP

📌 Chunking — phrase-level (IOB2 format):
  Token           Chunk Tag    Phrase Type
  ------------------------------------------
  John            B-NP         Noun Phrase
  works           B-VP         Verb Phrase
  at              B-PP         Prepositional Phrase
  Google          B-NP         Noun Phrase
  in              B-PP         Prepositional Phrase
  California      B-NP         Noun Phrase

🔍 Key Insight:
  POS Tagging → WHAT is each word grammatically? (noun, verb, adjective…)
  Chunking    → HOW do words group into phrases? (NP, VP, PP…)


---
## ✅ Task 8: Report

### 📝 NLP Deep Dive: POS Tagging vs Chunking with Transformer Models

---

#### 🔷 1. Introduction
In this assignment, I fine-tuned **DistilBERT** for two core NLP token classification tasks:
**POS Tagging** and **Chunking**, using the CoNLL-2000 dataset from NLTK.

---

#### 🔷 2. Differences Between POS Tagging and Chunking

| Feature | POS Tagging | Chunking |
|---|---|---|
| **Goal** | Label each word's grammatical role | Group words into phrases |
| **Granularity** | Word-level | Phrase-level |
| **Format** | Flat tags (NN, VB, JJ) | IOB2 (B-NP, I-NP, O) |
| **Complexity** | Easier | Moderate |
| **Example** | `John/NNP works/VBZ` | `[John]/NP [works]/VP` |

---

#### 🔷 3. Challenges Faced

**Challenge 1: Subword Tokenization & Label Alignment**
BERT uses WordPiece tokenization. A single word like `"playing"` becomes `["play", "##ing"]`.
Resolved by assigning the real label to the first subword only, and `-100` to the rest.

**Challenge 2: KeyError on Unseen Test Labels**
The test set contained chunk tags not present in training data.
Fixed by building `label_list` from **both train and test** before encoding.

**Challenge 3: Deprecated API Arguments**
- `evaluation_strategy` → renamed to `eval_strategy` in `transformers >= 4.41`
- `tokenizer=` in `Trainer()` → renamed to `processing_class=` in `transformers >= 4.46`
- `trust_remote_code=True` → no longer supported in newer `datasets` versions

**Challenge 4: IOB2 Format for Chunking**
`seqeval` evaluates chunks as complete units — a chunk is correct only if ALL its tokens
are correctly labeled, making chunking harder to score than POS tagging.

---

#### 🔷 4. Observations and Insights

- DistilBERT achieves strong results on both tasks with just 3 epochs — demonstrating the power of transfer learning.
- POS Tagging is easier because labels are independent per token; Chunking requires understanding phrase boundaries.
- The `seqeval` metric is stricter than token-level accuracy.
- Label alignment is the most critical preprocessing step — errors here affect the entire pipeline.
- Fine-tuning a pre-trained model gives far better results than training from scratch, especially with limited data.

---

#### 🔷 5. Conclusion

This assignment provided deep hands-on experience with transformer-based token classification.
The combination of Hugging Face Transformers, seqeval, and PyTorch makes the end-to-end pipeline
from data loading to evaluation to inference efficient and reproducible.

---
## 📝 Cell 22 — Pipeline Summary

In [23]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║            FULL PIPELINE SUMMARY — NLP Task 5                       ║
╠══════════════════════════════════════════════════════════════════════╣
║  1. Raw Data       → CoNLL-2000 via NLTK (tokens + labels)          ║
║  2. Tokenization   → DistilBERT WordPiece tokenizer                 ║
║  3. Label Align    → -100 for subwords & special tokens             ║
║  4. Model          → AutoModelForTokenClassification (DistilBERT)   ║
║  5. Training       → Hugging Face Trainer (3 epochs, LR=2e-5)       ║
║  6. Evaluation     → seqeval (Precision / Recall / F1)              ║
║  7. Inference      → Pipeline + Manual prediction on custom text    ║
║  8. Comparison     → POS Tagging (Easy) vs Chunking (Medium)        ║
╚══════════════════════════════════════════════════════════════════════╝
""")
print('✅ NLP Task 5 — COMPLETE!')


╔══════════════════════════════════════════════════════════════════════╗
║            FULL PIPELINE SUMMARY — NLP Task 5                       ║
╠══════════════════════════════════════════════════════════════════════╣
║  1. Raw Data       → CoNLL-2000 via NLTK (tokens + labels)          ║
║  2. Tokenization   → DistilBERT WordPiece tokenizer                 ║
║  3. Label Align    → -100 for subwords & special tokens             ║
║  4. Model          → AutoModelForTokenClassification (DistilBERT)   ║
║  5. Training       → Hugging Face Trainer (3 epochs, LR=2e-5)       ║
║  6. Evaluation     → seqeval (Precision / Recall / F1)              ║
║  7. Inference      → Pipeline + Manual prediction on custom text    ║
║  8. Comparison     → POS Tagging (Easy) vs Chunking (Medium)        ║
╚══════════════════════════════════════════════════════════════════════╝

✅ NLP Task 5 — COMPLETE!
